## Week 2 exercise — orchestrator → clarify → research → evaluate

[`ai_agents.py`](./ai_agents.py) defines specialist agents plus **`orchestrator_session_step`**, which drives the flow:

1. **Intent** (`orchestrator_intent_agent`) — research vs small talk  
2. **Clarify** — if research, ask clarifying questions (or skip if none)  
3. **Deep research** — web search + summary  
4. **Evaluate** — check answer vs request + clarifying choice  
5. **`orchestrator_agent`** — one user-facing message with findings and verdict  

Run from this folder. Set `OPENAI_API_KEY`. Web search is billed by OpenAI.

Below: **programmatic turns** (cells), then **Gradio** in separate steps (imports → handlers → UI → launch).

### Programmatic demo — setup

In [ ]:
from ai_agents import OrchestratorSession, orchestrator_session_step
from dotenv import load_dotenv
from agents import trace
from IPython.display import display, Markdown

load_dotenv(override=True)

### Turn 1 — user asks for research

In [ ]:
session = OrchestratorSession()

with trace("Turn 1"):
    session, reply = await orchestrator_session_step(
        session,
        "I want to compare cheap ways to host a small Python API with minimal ops.",
    )

display(Markdown(reply))

### Turn 2 — user’s clarifying choice

In [ ]:
with trace("Turn 2"):
    session, reply = await orchestrator_session_step(
        session,
        "Budget under $50/mo, prefer managed/serverless, low traffic hobby project.",
    )

display(Markdown(reply))

### Optional — small talk

In [ ]:
session2 = OrchestratorSession()
with trace("Small talk"):
    session2, reply2 = await orchestrator_session_step(session2, "Hey, how are you?")
display(Markdown(reply2))

---

## Gradio UI — Step 1: imports

Run the next cells in order. The last cell opens the app in your browser (stop the kernel to shut it down).

In [ ]:
from __future__ import annotations

import os

import gradio as gr
from dotenv import load_dotenv

from ai_agents import OrchestratorSession, orchestrator_session_step

load_dotenv(override=True)

### Gradio — Step 2: event handlers

In [ ]:
async def on_message(
    message: str,
    history: list | None,
    session: OrchestratorSession | None,
) -> tuple[list, OrchestratorSession, str]:
    text = (message or "").strip()
    if not text:
        return history or [], session or OrchestratorSession(), ""

    hist = list(history or [])
    sess = session if session is not None else OrchestratorSession()

    new_session, reply = await orchestrator_session_step(sess, text)
    hist.append({"role": "user", "content": text})
    hist.append({"role": "assistant", "content": reply})
    return hist, new_session, ""


async def on_reset() -> tuple[list, OrchestratorSession, str]:
    return [], OrchestratorSession(), ""


def _check_key() -> str | None:
    if not os.environ.get("OPENAI_API_KEY"):
        return (
            "**Warning:** `OPENAI_API_KEY` is not set. Add it to your `.env` or environment "
            "before sending messages."
        )
    return None


def _warn_on_load():
    w = _check_key()
    if w:
        return gr.update(visible=True, value=w)
    return gr.update(visible=False)

### Gradio — Step 3: build the interface

In [ ]:
with gr.Blocks(title="Research orchestrator") as demo:
    gr.Markdown(
        "## Research orchestrator\n\n"
        "Chat with the assistant. Ask for **research** (comparisons, facts, options) to run "
        "clarifying questions, then web search, evaluation, and a composed reply. "
        "Casual messages get a short chat response.\n\n"
        "*Web search usage is billed by OpenAI.*"
    )
    warn = gr.Markdown(visible=False)
    session_state = gr.State(OrchestratorSession())
    chatbot = gr.Chatbot(
        label="Conversation",
        type="messages",
        height=440,
        show_copy_button=True,
    )
    msg = gr.Textbox(
        label="Your message",
        placeholder="e.g. Compare managed hosts for a small Python API…",
        lines=2,
    )
    with gr.Row():
        send = gr.Button("Send", variant="primary")
        reset = gr.Button("Reset conversation")

    demo.load(_warn_on_load, outputs=warn)

    send.click(
        on_message,
        inputs=[msg, chatbot, session_state],
        outputs=[chatbot, session_state, msg],
    )
    msg.submit(
        on_message,
        inputs=[msg, chatbot, session_state],
        outputs=[chatbot, session_state, msg],
    )
    reset.click(
        on_reset,
        inputs=None,
        outputs=[chatbot, session_state, msg],
    )

### Gradio — Step 4: launch

Starts a local server; open the printed URL. Re-run this cell if you changed the UI cell above.

In [ ]:
demo.launch()